In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import math
import random
from tqdm import tqdm

# For notebook usage
try:
    from IPython.display import clear_output
except ImportError:
    pass

# Configuration
CONFIG = {
    "VOLUME_PATH": "sliepner.npy",
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "BATCH_SIZE": 4,
    "IMG_SIZE": 256,
    "UNET_DIM": 32,
    "AE_DIM": 32,
    "SEED": 42,
    "DIFFUSION_STEPS": 1000,
    "EXTRACTION_TIMESTEP": 100, # The timestep to extract features at (t=100 is good for structure)
    "TRAIN_EPOCHS": 25,          
    "LR": 1e-3,
    "N_VISUALIZATION_EXAMPLES": 3 # Number of examples to plot inline
}

# Reproducibility
random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
torch.manual_seed(CONFIG["SEED"])

# ==========================================
# 1. Real Seismic Data Loader
# ==========================================
class RealSeismicDataset(Dataset):
    def __init__(self, volume_path, img_size=64):
        if not os.path.exists(volume_path):
            print(f"Warning: {volume_path} not found. Generating structured synthetic volume.")
            # Create a dummy volume [Depth, Height, Width]
            self.vol = np.random.randn(100, img_size, img_size).astype(np.float32) * 0.1
            # Add synthetic geology: Layers and Faults
            for i in range(100):
                # Create layers
                freq = 0.2 + (i % 3) * 0.1
                for r in range(img_size):
                    self.vol[i, r, :] += np.sin(r * freq) 
                
                # Add a "fault" in some slices
                if i > 50:
                    self.vol[i, :, 32:] = np.roll(self.vol[i, :, 32:], 5, axis=0)
        else:
            print(f"Loading volume from {volume_path}...")
            self.vol = np.load(volume_path).astype(np.float32)

        # Robust Normalization to [-1, 1]
        robust_max = np.percentile(np.abs(self.vol), 99)
        if robust_max < 1e-6: robust_max = 1.0
        self.vol = np.clip(self.vol / robust_max, -1, 1)
        
        self.img_size = img_size

    def __len__(self):
        return self.vol.shape[0]

    def __getitem__(self, idx):
        img = self.vol[idx]
        h, w = img.shape
        img_t = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0) # [1, 1, H, W]
        
        if h != self.img_size or w != self.img_size:
            img_t = F.interpolate(img_t, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
            
        return img_t.squeeze(0) # Returns [1, H, W]

# ==========================================
# 2. The Autoencoder Model
# ==========================================
class SimpleAutoencoder(nn.Module):
    def __init__(self, dim=32):
        super().__init__()
        
        # Calculate feature map size after 3 downsampling layers (reduction by 2^3 = 8)
        self.f_dim = CONFIG["IMG_SIZE"] // 8  
        self.flat_dim = dim * 4 * self.f_dim * self.f_dim

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, dim, 4, 2, 1), # /2
            nn.ReLU(),
            nn.Conv2d(dim, dim*2, 4, 2, 1), # /4
            nn.ReLU(),
            nn.Conv2d(dim*2, dim*4, 4, 2, 1), # /8
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(self.flat_dim, 128) # Bottleneck
        )
        
        # Decoder
        self.decoder_fc = nn.Linear(128, self.flat_dim)
        self.decoder_cnn = nn.Sequential(
            nn.ConvTranspose2d(dim*4, dim*2, 4, 2, 1), # *2
            nn.ReLU(),
            nn.ConvTranspose2d(dim*2, dim, 4, 2, 1), # *4
            nn.ReLU(),
            nn.ConvTranspose2d(dim, 1, 4, 2, 1), # *8
            nn.Tanh()
        )
        self.dim = dim

    def forward(self, x):
        embedding = self.encoder(x)
        x_recon = self.decoder_fc(embedding)
        x_recon = x_recon.view(-1, self.dim*4, self.f_dim, self.f_dim)
        x_recon = self.decoder_cnn(x_recon)
        return x_recon, embedding

# ==========================================
# 3. The Diffusion U-Net & Utilities
# ==========================================

class DiffusionUtils:
    def __init__(self, steps=1000, device="cuda"):
        self.steps = steps
        self.device = device
        
        self.betas = torch.linspace(0.0001, 0.02, steps).to(device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)

    def q_sample(self, x_0, t, noise=None):
        if noise is None: noise = torch.randn_like(x_0)
        return (self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1) * x_0 + 
                self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1) * noise), noise

    def predict_start_from_noise(self, x_t, t, noise):
        """
        Recover x_0 from x_t and predicted noise.
        x_0 = (x_t - sqrt(1-alpha_cumprod)*noise) / sqrt(alpha_cumprod)
        """
        return (
            x_t - self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1) * noise
        ) / self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)

class Block(nn.Module):
    def __init__(self, dim, dim_out, groups=8):
        super().__init__()
        self.proj = nn.Conv2d(dim, dim_out, 3, padding=1)
        self.norm = nn.GroupNorm(groups, dim_out)
        self.act = nn.SiLU()
    def forward(self, x, scale_shift=None):
        x = self.proj(x)
        x = self.norm(x)
        if scale_shift is not None:
            scale, shift = scale_shift
            x = x * (scale + 1) + shift
        return self.act(x)

class ResnetBlock(nn.Module):
    def __init__(self, dim, dim_out, *, time_emb_dim=None, groups=8):
        super().__init__()
        self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, dim_out * 2)) if time_emb_dim else None
        self.block1 = Block(dim, dim_out, groups=groups)
        self.block2 = Block(dim_out, dim_out, groups=groups)
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()
    def forward(self, x, time_emb=None):
        scale_shift = None
        if self.mlp is not None and time_emb is not None:
            time_emb = self.mlp(time_emb)
            time_emb = time_emb.unsqueeze(-1).unsqueeze(-1)
            scale_shift = time_emb.chunk(2, dim=1)
        h = self.block1(x, scale_shift=scale_shift)
        h = self.block2(h)
        return h + self.res_conv(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        hidden_dim = dim_head * heads
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias=False)
        self.to_out = nn.Conv2d(hidden_dim, dim, 1)
    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.to_qkv(x).chunk(3, dim=1)
        q, k, v = map(lambda t: t.reshape(b, self.heads, -1, h * w), qkv)
        q = q * self.scale
        sim = torch.einsum('b h d i, b h d j -> b h i j', q, k)
        attn = sim.softmax(dim=-1)
        out = torch.einsum('b h i j, b h d j -> b h d i', attn, v)
        out = out.reshape(b, -1, h, w)
        return self.to_out(out) + x

class UNet(nn.Module):
    def __init__(self, dim=CONFIG["UNET_DIM"], out_dim=1, channels=1): 
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(dim), nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim * 4),
        )
        self.init_conv = nn.Conv2d(channels, dim, 7, padding=3)
        self.downs = nn.ModuleList([
            ResnetBlock(dim, dim, time_emb_dim=dim*4),
            ResnetBlock(dim, dim*2, time_emb_dim=dim*4),
            ResnetBlock(dim*2, dim*4, time_emb_dim=dim*4)
        ])
        self.down_samples = nn.ModuleList([nn.Conv2d(dim, dim, 4, 2, 1), nn.Conv2d(dim*2, dim*2, 4, 2, 1), nn.Conv2d(dim*4, dim*4, 4, 2, 1)])
        
        self.mid_block1 = ResnetBlock(dim*4, dim*4, time_emb_dim=dim*4)
        self.mid_attn = Attention(dim*4) 
        self.mid_block2 = ResnetBlock(dim*4, dim*4, time_emb_dim=dim*4)
        
        self.ups = nn.ModuleList([
            ResnetBlock(dim*8, dim*2, time_emb_dim=dim*4),
            ResnetBlock(dim*4, dim, time_emb_dim=dim*4),
            ResnetBlock(dim*2, dim, time_emb_dim=dim*4)
        ])
        self.up_samples = nn.ModuleList([nn.ConvTranspose2d(dim*4, dim*4, 4, 2, 1), nn.ConvTranspose2d(dim*2, dim*2, 4, 2, 1), nn.ConvTranspose2d(dim, dim, 4, 2, 1)])
        self.final_conv = nn.Conv2d(dim, out_dim, 1)

    def forward(self, x, time, return_embedding=False):
        t = self.time_mlp(time)
        x = self.init_conv(x)
        h_vals = []
        
        for block, sample in zip(self.downs, self.down_samples):
            x = block(x, t)
            h_vals.append(x)
            x = sample(x)
            
        # Bottleneck (Rich Embedding)
        x = self.mid_block1(x, t)
        x = self.mid_attn(x)
        x = self.mid_block2(x, t)
        
        if return_embedding:
            # Global Average Pooling [B, C, H, W] -> [B, C]
            return torch.mean(x, dim=(2, 3))
            
        for block, sample in zip(self.ups, self.up_samples):
            x = sample(x)
            h = h_vals.pop()
            if x.shape[2:] != h.shape[2:]: 
                x = F.interpolate(x, size=h.shape[2:], mode='nearest')
            x = torch.cat((x, h), dim=1)
            x = block(x, t)
            
        return self.final_conv(x)

# ==========================================
# 4. Training Loops
# ==========================================

def train_ae(model, loader, epochs):
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LR"])
    criterion = nn.MSELoss()
    model.train()
    
    print(f"\nTraining Autoencoder for {epochs} epochs...")
    
    for epoch in range(epochs):
        total_loss = 0
        for imgs in tqdm(loader, desc=f"AE Epoch {epoch+1}", leave=False):
            imgs = imgs.to(CONFIG["DEVICE"])
            optimizer.zero_grad()
            recon, _ = model(imgs)
            loss = criterion(recon, imgs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"AE Epoch {epoch+1}: Loss {total_loss/len(loader):.5f}")

def train_diffusion(model, diff_utils, loader, epochs):
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LR"])
    criterion = nn.MSELoss()
    model.train()
    
    print(f"\nTraining Diffusion for {epochs} epochs...")
    
    for epoch in range(epochs):
        total_loss = 0
        for imgs in tqdm(loader, desc=f"Diff Epoch {epoch+1}", leave=False):
            imgs = imgs.to(CONFIG["DEVICE"])
            b = imgs.shape[0]
            
            t = torch.randint(0, diff_utils.steps, (b,), device=CONFIG["DEVICE"]).long()
            noise = torch.randn_like(imgs)
            noisy_imgs, _ = diff_utils.q_sample(imgs, t, noise)
            
            optimizer.zero_grad()
            pred_noise = model(noisy_imgs, t)
            loss = criterion(pred_noise, noise)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        print(f"Diff Epoch {epoch+1}: Loss {total_loss/len(loader):.5f}")

# ==========================================
# 5. Analysis & Visualization
# ==========================================

def get_embeddings_and_labels(dataset, ae_model, diff_model, diff_utils):
    loader = DataLoader(dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False)
    ae_model.eval()
    diff_model.eval()
    
    ae_embs = []
    diff_embs = []
    
    # We also keep track of the original images to pick examples later
    print("\nExtracting embeddings...")
    with torch.no_grad():
        for imgs in tqdm(loader, desc="Extracting"):
            imgs = imgs.to(CONFIG["DEVICE"])
            
            # AE Embedding
            _, ae_z = ae_model(imgs)
            ae_embs.append(ae_z.cpu().numpy())
            
            # Diffusion Embedding
            b = imgs.shape[0]
            t = torch.full((b,), CONFIG["EXTRACTION_TIMESTEP"], device=CONFIG["DEVICE"]).long()
            noise = torch.randn_like(imgs)
            noisy_imgs, _ = diff_utils.q_sample(imgs, t, noise)
            diff_z = diff_model(noisy_imgs, t, return_embedding=True)
            diff_embs.append(diff_z.cpu().numpy())
            
    return np.concatenate(ae_embs), np.concatenate(diff_embs)

def visualize_results(dataset, ae_model, diff_model, diff_utils):
    # 1. Get Embeddings
    ae_emb, diff_emb = get_embeddings_and_labels(dataset, ae_model, diff_model, diff_utils)
    
    # 2. PCA
    print("Computing PCA...")
    pca_ae = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(ae_emb))
    pca_diff = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(diff_emb))
    
    # 3. K-Means
    print("Clustering...")
    kmeans_ae = KMeans(n_clusters=3, random_state=42).fit(ae_emb)
    kmeans_diff = KMeans(n_clusters=3, random_state=42).fit(diff_emb)
    
    labels_ae = kmeans_ae.labels_
    labels_diff = kmeans_diff.labels_
    
    # 4. PLOT 1: Embedding Space
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    scatter1 = axes[0].scatter(pca_ae[:, 0], pca_ae[:, 1], c=labels_ae, cmap='viridis', edgecolors='k')
    axes[0].set_title("Autoencoder Embeddings (PCA)")
    axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
    
    scatter2 = axes[1].scatter(pca_diff[:, 0], pca_diff[:, 1], c=labels_diff, cmap='viridis', edgecolors='k')
    axes[1].set_title("Diffusion Embeddings (PCA)")
    axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
    
    plt.suptitle("Embedding Space Comparison", fontsize=16)
    plt.show()
    
    # 5. PLOT 2: Inline Examples (Slice + Models + Labels)
    print(f"\nPlotting {CONFIG['N_VISUALIZATION_EXAMPLES']} random examples inline...")
    
    indices = np.random.choice(len(dataset), CONFIG["N_VISUALIZATION_EXAMPLES"], replace=False)
    
    # Prepare figure for examples: 3 columns (Original, AE Recon, Diff Approx)
    fig, axs = plt.subplots(CONFIG["N_VISUALIZATION_EXAMPLES"], 3, figsize=(12, 4 * CONFIG["N_VISUALIZATION_EXAMPLES"]))
    
    ae_model.eval()
    diff_model.eval()
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            # Load single item
            img_t = dataset[idx].unsqueeze(0).to(CONFIG["DEVICE"]) # [1, 1, H, W]
            
            # --- Autoencoder Path ---
            ae_recon, _ = ae_model(img_t)
            ae_cluster = labels_ae[idx]
            
            # --- Diffusion Path ---
            # To show what the diffusion model "sees", we noise it to t=100 and predict x0
            t_val = CONFIG["EXTRACTION_TIMESTEP"]
            t = torch.tensor([t_val], device=CONFIG["DEVICE"]).long()
            noise = torch.randn_like(img_t)
            noisy_img, _ = diff_utils.q_sample(img_t, t, noise)
            
            pred_noise = diff_model(noisy_img, t)
            diff_recon = diff_utils.predict_start_from_noise(noisy_img, t, pred_noise)
            diff_cluster = labels_diff[idx]
            
            # --- Visualization ---
            # 1. Original
            im1 = axs[i, 0].imshow(img_t.cpu().squeeze().T, cmap='gray')
            axs[i, 0].set_title(f"Original Slice #{idx}")
            axs[i, 0].axis('off')
            
            # 2. AE Recon
            im2 = axs[i, 1].imshow(ae_recon.cpu().squeeze().detach().T, cmap='gray')
            axs[i, 1].set_title(f"AE Recon (Cluster {ae_cluster})")
            axs[i, 1].axis('off')
            
            # 3. Diffusion Approx
            im3 = axs[i, 2].imshow(diff_recon.cpu().squeeze().detach().T, cmap='gray')
            axs[i, 2].set_title(f"Diff Denoised t={t_val} (Cluster {diff_cluster})")
            axs[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig('seismic_examples.png')
    plt.show()
    print("Visualization saved to 'seismic_examples.png'")

# ==========================================
# 6. Main
# ==========================================

def run_experiment():
    # Setup
    dataset = RealSeismicDataset(CONFIG["VOLUME_PATH"], img_size=CONFIG["IMG_SIZE"])
    loader = DataLoader(dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, drop_last=True)
    
    diff_utils = DiffusionUtils(steps=CONFIG["DIFFUSION_STEPS"], device=CONFIG["DEVICE"])
    
    ae = SimpleAutoencoder(dim=CONFIG["AE_DIM"]).to(CONFIG["DEVICE"])
    unet = UNet(dim=CONFIG["UNET_DIM"]).to(CONFIG["DEVICE"])
    
    # Train
    train_ae(ae, loader, epochs=CONFIG["TRAIN_EPOCHS"])
    train_diffusion(unet, diff_utils, loader, epochs=CONFIG["TRAIN_EPOCHS"])
    
    # Visualize
    visualize_results(dataset, ae, unet, diff_utils)

if __name__ == "__main__":
    run_experiment()